# Roboflow Food Ingredients Model Setup

**Purpose**: Setup and test Roboflow pre-trained food ingredients detection model using Inference SDK

**Task**: T012-T014 [P] [US1] (Consolidated)

**Model Source**: 
- Roboflow Food Ingredients Dataset v2
- URL: https://universe.roboflow.com/food-recipe-ingredient-images-0gnku/food-ingredients-dataset/

**Outputs**:
- Model configuration saved to `models/ingredient_recognition/`
- Inference client ready for use
- Test inference results

## 1. Environment Setup

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import json

# Roboflow Inference SDK
try:
    from inference_sdk import InferenceHTTPClient
    print("✅ Inference SDK installed")
except ImportError:
    print("⚠️ Installing Inference SDK...")
    !pip install inference-sdk
    from inference_sdk import InferenceHTTPClient
    print("✅ Inference SDK installed")

# Set random seed
np.random.seed(42)

print("✅ Packages imported successfully")

✅ Inference SDK installed
✅ Packages imported successfully


## 2. Configure Paths and API

In [2]:
# Project directories
PROJECT_ROOT = Path.cwd().parent.parent
MODEL_DIR = PROJECT_ROOT / "models" / "ingredient_recognition"
DATA_TEST = PROJECT_ROOT / "data" / "test_images"

# Create directories
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DATA_TEST.mkdir(parents=True, exist_ok=True)

# Roboflow configuration
ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', 'kuzgSqDiqDLJkXrzcqRr')
API_URL = "https://serverless.roboflow.com"
MODEL_ID = "food-ingredients-dataset/2"

print(f"📁 Model directory: {MODEL_DIR}")
print(f"📁 Test images: {DATA_TEST}")
print(f"🔑 API Key: {ROBOFLOW_API_KEY[:10]}...")
print(f"🌐 API URL: {API_URL}")
print(f"🎯 Model ID: {MODEL_ID}")

📁 Model directory: c:\Users\Champion\Documents\GitHub\cAIuldron\models\ingredient_recognition
📁 Test images: c:\Users\Champion\Documents\GitHub\cAIuldron\data\test_images
🔑 API Key: kuzgSqDiqD...
🌐 API URL: https://serverless.roboflow.com
🎯 Model ID: food-ingredients-dataset/2


## 3. Initialize Inference Client

In [3]:
# Initialize Roboflow Inference Client
CLIENT = InferenceHTTPClient(
    api_url=API_URL,
    api_key=ROBOFLOW_API_KEY
)

print("✅ Roboflow Inference Client initialized")
print(f"\n📊 Client Configuration:")
print(f"  - API URL: {API_URL}")
print(f"  - Model: {MODEL_ID}")
print(f"  - Version: 2")
print("\n✅ Ready for inference!")

✅ Roboflow Inference Client initialized

📊 Client Configuration:
  - API URL: https://serverless.roboflow.com
  - Model: food-ingredients-dataset/2
  - Version: 2

✅ Ready for inference!


## 4. Test Inference Function

In [4]:
def test_inference(image_path):
    """
    Test inference on an image
    
    Args:
        image_path: Path to test image
    
    Returns:
        dict: Inference results
    """
    try:
        result = CLIENT.infer(str(image_path), model_id=MODEL_ID)
        return result
    except Exception as e:
        print(f"❌ Inference failed: {e}")
        return None

print("✅ Test inference function defined")

✅ Test inference function defined


## 5. Test with Sample Image (if available)

In [5]:
# Check for test images
test_images = list(DATA_TEST.glob("*.jpg")) + list(DATA_TEST.glob("*.png")) + list(DATA_TEST.glob("*.webp"))

if test_images:
    print(f"✅ Found {len(test_images)} test images")
    
    # Test with first image
    test_image = test_images[0]
    print(f"\n🧪 Testing with: {test_image.name}")
    
    # Run inference
    result = test_inference(test_image)
    
    if result:
        print("\n✅ Inference successful!")
        print(f"\n📊 Result structure:")
        print(f"  - Keys: {list(result.keys())}")
        
        if 'predictions' in result:
            print(f"  - Predictions: {len(result['predictions'])}")
            
            if result['predictions']:
                print(f"\n🎯 Top prediction:")
                top = result['predictions'][0]
                print(f"  - Class: {top.get('class', 'N/A')}")
                print(f"  - Confidence: {top.get('confidence', 0):.2%}")
                print(f"  - Bounding box: ({top.get('x', 0):.0f}, {top.get('y', 0):.0f}) "
                      f"W={top.get('width', 0):.0f} H={top.get('height', 0):.0f}")
else:
    print(f"⚠️ No test images found in {DATA_TEST}")
    print("\nPlease add test images (raw ingredients) to test the model")
    print("\nExample usage:")
    print("```python")
    print("result = CLIENT.infer('path/to/ingredient_photo.jpg', model_id='food-ingredients-dataset/2')")
    print("```")

✅ Found 2 test images

🧪 Testing with: eyecatch-4569.jpg

✅ Inference successful!

📊 Result structure:
  - Keys: ['inference_id', 'time', 'image', 'predictions']
  - Predictions: 1

🎯 Top prediction:
  - Class: Akabare Khursani
  - Confidence: 56.26%
  - Bounding box: (538, 175) W=80 H=70


## 6. Extract Model Information from Response

In [6]:
# Extract model info from a sample inference (if available)
if test_images and 'result' in locals() and result:
    # Extract image dimensions
    img_info = {
        'image_width': result.get('image', {}).get('width', None),
        'image_height': result.get('image', {}).get('height', None)
    }
    
    # Extract class names from predictions
    classes = set()
    if 'predictions' in result:
        for pred in result['predictions']:
            if 'class' in pred:
                classes.add(pred['class'])
    
    print("📊 Model Information:")
    print(f"  - Input image: {img_info['image_width']}x{img_info['image_height']}")
    print(f"  - Detected classes in this image: {list(classes)}")
else:
    print("💡 Upload test images to extract model information")

📊 Model Information:
  - Input image: 660x400
  - Detected classes in this image: ['Akabare Khursani']


## 7. Save Configuration

In [7]:
# Save model configuration
config = {
    'model_source': 'Roboflow Serverless',
    'model_id': MODEL_ID,
    'version': 2,
    'api_url': API_URL,
    'deployment_mode': 'serverless',  # Using Roboflow serverless API
    'confidence_threshold': 0.7,
    'overlap_threshold': 0.3,
    'sdk': 'inference-sdk',
    'notes': [
        'Using Roboflow Inference SDK for lightweight, fast inference',
        'Serverless deployment - no local model download needed',
        'Model detects raw food ingredients (meat, vegetables, fruits)',
        'Returns bounding boxes, class labels, and confidence scores',
        'Version 2 of food-ingredients-dataset'
    ]
}

config_file = MODEL_DIR / 'model_config.json'
with open(config_file, 'w') as f:
    json.dump(config, f, indent=2)

print(f"✅ Configuration saved to: {config_file}")
print("\n📋 Configuration:")
print(json.dumps(config, indent=2))

✅ Configuration saved to: c:\Users\Champion\Documents\GitHub\cAIuldron\models\ingredient_recognition\model_config.json

📋 Configuration:
{
  "model_source": "Roboflow Serverless",
  "model_id": "food-ingredients-dataset/2",
  "version": 2,
  "api_url": "https://serverless.roboflow.com",
  "deployment_mode": "serverless",
  "confidence_threshold": 0.7,
  "overlap_threshold": 0.3,
  "sdk": "inference-sdk",
  "notes": [
    "Using Roboflow Inference SDK for lightweight, fast inference",
    "Serverless deployment - no local model download needed",
    "Model detects raw food ingredients (meat, vegetables, fruits)",
    "Returns bounding boxes, class labels, and confidence scores",
    "Version 2 of food-ingredients-dataset"
  ]
}


## 8. Create Helper Functions

In [ ]:
def get_client():
    """
    Get configured Roboflow Inference Client
    
    Returns:
        InferenceHTTPClient: Configured client
    """
    return InferenceHTTPClient(
        api_url=API_URL,
        api_key=ROBOFLOW_API_KEY
    )

def infer_ingredient(image_path, confidence=70, overlap=30):
    """
    Detect ingredients in image
    
    Args:
        image_path: Path to image
        confidence: Confidence threshold (0-100)
        overlap: Overlap threshold for NMS (0-100)
    
    Returns:
        dict: Inference results
    """
    client = get_client()
    return client.infer(
        str(image_path),
        model_id=MODEL_ID
    )

print("✅ Helper functions defined")
print("\n💡 Usage:")
print("   result = infer_ingredient('path/to/image.jpg')")

## 9. Summary

### ✅ Completed:
1. ✅ Installed Inference SDK
2. ✅ Initialized Roboflow Inference Client
3. ✅ Configured serverless API connection
4. ✅ Tested inference (if test images available)
5. ✅ Saved model configuration
6. ✅ Created helper functions

### 🎯 Model Details:
- **Model**: food-ingredients-dataset/2
- **Type**: Object Detection
- **Deployment**: Roboflow Serverless
- **SDK**: inference-sdk (lightweight)

### 💡 Advantages:
- ✅ **No model download** - Uses serverless API
- ✅ **Lightweight** - Only need inference-sdk package
- ✅ **Fast** - Direct HTTP inference
- ✅ **Simple** - One-line inference call
- ✅ **Maintained** - Roboflow handles updates

### 📝 Next Steps:
1. Add test images to `data/test_images/`
2. Use inference in `model_cnn_inference.ipynb`
3. Integrate with recipe generation pipeline

In [ ]:
print("🎉 Roboflow Inference SDK setup complete!")
print(f"\n📁 Configuration: {config_file}")
print(f"🎯 Model: {MODEL_ID}")
print("\n✅ Ready for ingredient detection!")